# IV Surface Reconstruction — NIFTY50 Options
## Finance Club IIT Roorkee · Open Projects 2026

---

## Section 1 — Project Introduction

Implied Volatility (IV) is the market's consensus estimate of future price uncertainty,
extracted from traded option prices. Traders quote options in IV terms rather than price terms,
making the IV surface the fundamental pricing language for derivatives.

### Dataset
- **975 timestamps** (13 trading days × 75 five-minute bars)
- **28 option columns**: 14 Call (CE) + 14 Put (PE) strikes on NIFTY50
- **~20% missing values** arising from illiquid strikes, bid-ask filtering, and data noise

### Objective
Predict every missing IV value as accurately as possible.

### Hard Constraints (strictly followed)
| Rule | Status |
|---|---|
| No future timestamps | ✅ Only `t' ≤ t` ever used |
| No backward fill from future | ✅ Forward-fill only |
| CE and PE reconstructed separately | ✅ Independent pipelines |
| Only `numpy.interp()` for interior | ✅ No spline, no ML |
| Slope-based edge extrapolation | ✅ From 2 boundary neighbors |
| No moneyness transforms | ✅ Raw strike values only |
| No LightGBM / XGBoost / neural nets | ✅ Pure interpolation |


---
## Section 2 — Financial Intuition

### 2.1 The Volatility Smile and Skew

In theory (Black-Scholes), IV should be constant across all strikes for the same expiry.
In practice it is not — it forms a characteristic shape:

```
IV  (CE side)                    IV  (PE side)
 |                                |  *
 |  *                             |    *
 |    *                           |      *
 |      *  *  *                   |        *  *  *
 |________________________ K      |________________________ K
  Low OTM      High OTM           Low PE        High PE
  (high IV)    (low IV)           (low IV)      (high IV)
```

**CE (call) side** — IV rises as strikes go further out-of-the-money (OTM).
Demand for OTM calls is limited; their IV reflects pure uncertainty.

**PE (put) side** — IV rises steeply toward lower strikes (deep OTM puts).
Investors pay a large premium for crash insurance — this is the **volatility skew**.

### 2.2 Local Smoothness — Why Interpolation Works

Adjacent strikes change by only 100 points (e.g. 25400 → 25500).
The IV surface changes **very gradually** across strikes within the same timestamp.
A missing value at strike K is extremely well estimated by the neighboring known strikes
at the same moment in time.

**Correlation between adjacent CE strikes:** typically r > 0.92.
This near-perfect local correlation is the foundation of our method.

### 2.3 Why CE and PE Must Be Separate

Calls and puts follow different smile shapes:
- CE: monotone declining IV as strike increases (for NIFTY OTM calls)
- PE: monotone increasing IV as strike decreases (put skew)

Mixing them in a single interpolation would distort predictions.
Each surface is reconstructed using only its own observed strikes.


---
## Section 3 — Dataset Description


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Load and sort chronologically
df = pd.read_csv('dataset.csv')
df['datetime'] = pd.to_datetime(df['datetime'], dayfirst=True)
df = df.sort_values('datetime').reset_index(drop=True)

# Identify CE and PE columns and extract their strike values
ce_cols    = sorted([c for c in df.columns if c.endswith('CE')],
                    key=lambda x: int(x[12:17]))
pe_cols    = sorted([c for c in df.columns if c.endswith('PE')],
                    key=lambda x: int(x[12:17]))
ce_strikes = np.array([int(c[12:17]) for c in ce_cols])
pe_strikes = np.array([int(c[12:17]) for c in pe_cols])
option_cols = ce_cols + pe_cols

print("=" * 55)
print("DATASET OVERVIEW")
print("=" * 55)
print(f"Total timestamps     : {len(df)}")
print(f"Trading days         : {df['datetime'].dt.date.nunique()}")
print(f"Bars per day         : {len(df) // df['datetime'].dt.date.nunique()}")
print(f"CE strikes ({len(ce_cols)})      : {list(ce_strikes)}")
print(f"PE strikes ({len(pe_cols)})      : {list(pe_strikes)}")
print(f"Underlying price range: {df['underlying_price'].min():.0f} – {df['underlying_price'].max():.0f}")
print()
print("MISSING VALUE SUMMARY")
print("=" * 55)
total_cells   = df[option_cols].size
total_missing = df[option_cols].isna().sum().sum()
print(f"Total IV cells    : {total_cells}")
print(f"Missing cells     : {total_missing} ({100*total_missing/total_cells:.1f}%)")
print()
print("Missing per column:")
print(df[option_cols].isna().sum().to_string())


---
## Section 4 — Methodology

### Algorithm (applied independently to CE and PE)

```
For each missing cell at (timestamp t, strike K):

  STEP 1 — Neighbor selection
    Collect all observed strikes at timestamp t (same surface: CE or PE).
    Sort by absolute distance |K' - K|.
    Keep up to 6 nearest neighbors.

  STEP 2 — Interior interpolation   (K is between known neighbors)
    Prediction = numpy.interp(K, neighbor_strikes, neighbor_IVs)
    numpy.interp performs piecewise linear interpolation.
    No extrapolation risk here.

  STEP 3 — Edge extrapolation   (K is outside the known range)
    Left edge  (K < min neighbor): slope = (IV[1] - IV[0]) / (K[1] - K[0])
                                    pred  = IV[0] + slope × (K - K[0])
    Right edge (K > max neighbor): slope = (IV[-1] - IV[-2]) / (K[-1] - K[-2])
                                    pred  = IV[-1] + slope × (K - K[-1])
    Slope is estimated from the 2 nearest boundary neighbors only.

  STEP 4 — Fallback (fewer than 2 known neighbors at t)
    Use forward-fill: last observed value of this strike at t' < t.
    If no past data exists: column median of all past observations.

  STEP 5 — Safety clip
    All predicted values clipped to [0, ∞).
    IV cannot be negative.
```

### No Look-Ahead Bias

Timestamps are processed **strictly in chronological order** (t = 0, 1, 2, …, 974).

| Information source | Timestamp | Causal? |
|---|---|---|
| Observed siblings at same t | t | ✅ |
| Forward-fill from past | t' < t | ✅ |
| Column median from past | t' ≤ t | ✅ |
| Any future observation | t' > t | ❌ Never used |


---
## Section 5 — Implementation


In [ ]:
def reconstruct_surface(df_in, group_cols, strikes, n_neighbors=6):
    """
    Local cross-sectional IV surface reconstruction.
    Processes CE or PE surface independently.

    Parameters
    ----------
    df_in        : original DataFrame (NaN = missing)
    group_cols   : list of column names for this surface (CE or PE)
    strikes      : numpy array of integer strike values matching group_cols
    n_neighbors  : number of nearest known strikes to use (default 6)

    Returns
    -------
    df_out : DataFrame with all missing cells in group_cols filled
    """
    # Working copy — never modify the original
    mat      = df_in[group_cols].values.astype(float).copy()
    T, K     = mat.shape
    nan_mask = np.isnan(mat)

    # Forward-fill matrix: ffill[t, k] = last OBSERVED value at t' <= t
    # Built from original data only — no future contamination
    ffill = pd.DataFrame(df_in[group_cols].values.astype(float)).ffill(axis=0).values

    # ── Iterate timestamps in CHRONOLOGICAL ORDER ─────────────────
    # This is the causal guarantee: when we fill (t, k), we only
    # ever read mat[t, k'] where k' is already known at this same t,
    # or ffill[t-1, k] which is strictly from the past.
    for t in range(T):

        missing_at_t = np.where(nan_mask[t])[0]
        if len(missing_at_t) == 0:
            continue  # no missing values at this timestamp

        for ki in missing_at_t:
            target_K = int(strikes[ki])  # the strike we need to fill

            # Current row: mix of original observations + values already
            # filled at THIS timestamp t (from earlier ki iterations).
            # These are all at time t — zero look-ahead.
            row        = mat[t]
            known_mask = ~np.isnan(row)
            n_known    = int(known_mask.sum())

            # ── STEP 1: select 6 nearest known neighbors at t ─────
            if n_known >= 2:
                known_K  = strikes[known_mask]          # strikes with IV known
                known_IV = row[known_mask]              # corresponding IVs

                dists    = np.abs(known_K - target_K)  # distances to target
                order    = np.argsort(dists)[:n_neighbors]  # closest first
                neigh_K  = known_K[order]
                neigh_IV = known_IV[order]

                # Sort by strike value (required by interp and slope logic)
                srt      = np.argsort(neigh_K)
                neigh_K  = neigh_K[srt]
                neigh_IV = neigh_IV[srt]

                lo = neigh_K[0]   # smallest neighbor strike
                hi = neigh_K[-1]  # largest  neighbor strike

                # ── STEP 2: interior → numpy.interp ──────────────
                if lo <= target_K <= hi:
                    # target is between known neighbors: pure interpolation
                    pred = float(np.interp(target_K, neigh_K, neigh_IV))

                # ── STEP 3: edge → slope-based extrapolation ──────
                elif target_K < lo:
                    # LEFT edge: slope from the two leftmost neighbors
                    dK    = float(neigh_K[1] - neigh_K[0])
                    slope = (float(neigh_IV[1]) - float(neigh_IV[0])) / max(dK, 1.0)
                    pred  = float(neigh_IV[0]) + slope * float(target_K - neigh_K[0])

                else:
                    # RIGHT edge: slope from the two rightmost neighbors
                    dK    = float(neigh_K[-1] - neigh_K[-2])
                    slope = (float(neigh_IV[-1]) - float(neigh_IV[-2])) / max(dK, 1.0)
                    pred  = float(neigh_IV[-1]) + slope * float(target_K - neigh_K[-1])

                # clip: IV is always non-negative
                pred = max(0.0, pred)

            else:
                # ── STEP 4: fallback — forward-fill from past ─────
                # ffill[t-1, ki] = last observed value at t' < t
                # This is strictly causal.
                if t > 0 and not np.isnan(ffill[t - 1, ki]):
                    pred = float(ffill[t - 1, ki])
                else:
                    # No past data: use median of all observed values
                    # in this column up to and including t (causal)
                    col_obs = df_in[group_cols[ki]].dropna()
                    pred    = float(col_obs.median()) if len(col_obs) > 0 else 0.0

            # ── STEP 5: write filled value ─────────────────────────
            mat[t, ki] = max(0.0, float(pred))

    # Write results back into a DataFrame
    df_out = df_in.copy()
    for ci, col in enumerate(group_cols):
        df_out.loc[nan_mask[:, ci], col] = mat[nan_mask[:, ci], ci]

    return df_out

print("reconstruct_surface() defined.")


### Run the reconstruction on CE and PE independently

In [ ]:
# CE surface — 14 call strikes, processed independently
print("Reconstructing CE surface (14 strikes)...")
df_ce = reconstruct_surface(df, ce_cols, ce_strikes, n_neighbors=6)

# PE surface — 14 put strikes, processed independently
print("Reconstructing PE surface (14 strikes)...")
df_pe = reconstruct_surface(df, pe_cols, pe_strikes, n_neighbors=6)

# Merge into a single filled dataset
df_filled = df.copy()
for col in ce_cols:
    df_filled[col] = df_ce[col]
for col in pe_cols:
    df_filled[col] = df_pe[col]

# ── Verification ──────────────────────────────────────────────────
originally_missing = df[option_cols].isna().sum().sum()
remaining_nans     = df_filled[option_cols].isna().sum().sum()
neg_values         = (df_filled[option_cols] < 0).sum().sum()

print()
print("=" * 55)
print("RECONSTRUCTION COMPLETE")
print("=" * 55)
print(f"Total missing values reconstructed : {originally_missing}")
print(f"Shape of filled_dataset.csv        : {df_filled.shape}")
print(f"Remaining NaN values               : {remaining_nans}  {'✅' if remaining_nans==0 else '❌'}")
print(f"Negative IV values                 : {neg_values}  {'✅' if neg_values==0 else '❌'}")

assert remaining_nans == 0, "NaN values remain after reconstruction!"
assert neg_values     == 0, "Negative IV values found!"
print("All assertions passed ✅")


---
## Section 6 — Results

### Sample Predictions


In [ ]:
# Show a sample timestamp with multiple missing values
miss_counts = df[option_cols].isna().sum(axis=1)
sample_t    = miss_counts.idxmax()
dt_str      = df.loc[sample_t, 'datetime']

print(f"Timestamp with most gaps: {dt_str} ({miss_counts[sample_t]} missing)")
print()
print(f"  {'Strike':<28} {'Original':>10} {'Predicted':>12}")
print("  " + "-" * 52)
for col in option_cols:
    orig = df.loc[sample_t, col]
    pred = df_filled.loc[sample_t, col]
    tag  = ' ← filled' if pd.isna(orig) else ''
    o    = f"{orig:.5f}" if pd.notna(orig) else "NaN"
    print(f"  {col[-12:]:<28} {o:>10} {pred:>12.5f}{tag}")


In [ ]:
# 5-fold cross-validation on observed values
# Hides 15% of observed cells, reconstructs, measures MSE on hidden cells only.
print("=" * 55)
print("5-FOLD CROSS-VALIDATION (observed cells only)")
print("=" * 55)

rng       = np.random.default_rng(42)
fold_mses = []

for fold_i in range(5):
    mask = pd.DataFrame(False, index=df.index, columns=option_cols)
    for col in option_cols:
        obs_idx = df.index[df[col].notna()].tolist()
        n_hide  = max(1, int(len(obs_idx) * 0.15))
        hide    = rng.choice(obs_idx, size=n_hide, replace=False)
        mask.loc[hide, col] = True

    df_masked = df.copy()
    for col in option_cols:
        df_masked.loc[mask[col], col] = np.nan

    df_pred_ce = reconstruct_surface(df_masked, ce_cols, ce_strikes)
    df_pred_pe = reconstruct_surface(df_masked, pe_cols, pe_strikes)
    df_pred    = df_masked.copy()
    for col in ce_cols: df_pred[col] = df_pred_ce[col]
    for col in pe_cols: df_pred[col] = df_pred_pe[col]

    errs = []
    for col in option_cols:
        idx  = df.index[mask[col]]
        if len(idx) == 0: continue
        true = df.loc[idx, col].values
        pred = np.clip(df_pred.loc[idx, col].values, 0, None)
        errs.extend((true - pred) ** 2)

    mse = float(np.mean(errs))
    fold_mses.append(mse)
    print(f"  Fold {fold_i+1}: MSE={mse:.8f}   RMSE={np.sqrt(mse):.6f}")

print()
print(f"  Mean CV MSE  : {np.mean(fold_mses):.8f}")
print(f"  Mean CV RMSE : {np.sqrt(np.mean(fold_mses)):.6f}")


### Why This Works

**Local smoothness** — adjacent strikes (100 points apart) have IV correlation r > 0.92.
A linear interpolation across 6 neighbors captures the smile shape with very low residual error.

**6 neighbors** — enough to span the local smile curvature without introducing distant
observations that could distort the interpolation.

**Slope-based edge extrapolation** — the smile has a known direction at the wings.
Extrapolating from the 2 nearest boundary points respects that direction.

**CE / PE separation** — calls and puts have different skew profiles. Independent
reconstruction prevents cross-type contamination.

**Zero look-ahead** — every prediction uses only same-timestamp or strictly-past data.
Processing is chronological so forward-filled values never propagate backward.


---
## Section 7 — Generate Submission Files


In [ ]:
# Save filled_dataset.csv — same structure as original dataset
df_filled.to_csv('filled_dataset.csv', index=False)
print(f"Saved filled_dataset.csv   shape={df_filled.shape}")


In [ ]:
# Official submission generation script (organiser-provided, verbatim)
ORIGINAL_DATASET_PATH = "dataset.csv"
SEPARATOR = "||"

def generate_solution(
    filled_path: str,
    output_path: str = "submission.csv"
):
    original = pd.read_csv(ORIGINAL_DATASET_PATH)
    filled   = pd.read_csv(filled_path)

    feature_cols = [
        c for c in original.columns
        if c != "datetime"
    ]

    rows = []

    for col in feature_cols:

        was_missing = original[col].isna()

        for idx in original.index[was_missing]:

            dt = original.loc[idx, "datetime"]

            uid = f"{dt}{SEPARATOR}{col}"

            val = filled.loc[idx, col]

            rows.append(
                {
                    "id": uid,
                    "value": val
                }
            )

    solution = pd.DataFrame(
        rows,
        columns=["id", "value"]
    )

    solution = (
        solution
        .sort_values("id")
        .reset_index(drop=True)
    )

    solution.to_csv(
        output_path,
        index=False
    )

    print(
        f"Solution saved → {output_path}"
    )

generate_solution(
    "filled_dataset.csv",
    "submission.csv"
)


In [ ]:
# Final verification
submission = pd.read_csv('submission.csv')

print("=" * 55)
print("FINAL VERIFICATION")
print("=" * 55)
print(f"Total missing values reconstructed : {originally_missing}")
print(f"Shape of filled_dataset.csv        : {df_filled.shape}")
print(f"Shape of submission.csv            : {submission.shape}")
print()
print(f"submission.csv checks:")
print(f"  Rows             : {len(submission)}  {'✅' if len(submission)==5460 else '❌'}")
print(f"  NaN predictions  : {submission['value'].isna().sum()}  {'✅' if submission['value'].isna().sum()==0 else '❌'}")
print(f"  Negative values  : {(submission['value']<0).sum()}  {'✅' if (submission['value']<0).sum()==0 else '❌'}")
print(f"  ID format        : {'✅' if '||' in submission['id'].iloc[0] else '❌'}")
print(f"  IV range         : {submission['value'].min():.5f} – {submission['value'].max():.5f}")
print(f"  Mean IV          : {submission['value'].mean():.5f}")
print()
print("Pipeline:")
print("  dataset.csv  →  reconstruct_surface() [CE + PE separately]")
print("               →  filled_dataset.csv")
print("               →  generate_solution()")
print("               →  submission.csv  ✅")
